# Model Sources: Unified Discovery, Download & Visualization

This notebook downloads a representative HEC-RAS model from each source, initializes it with ras-commander, and captures a RASMapper screenshot.

| Source | Representative Model | HEC-RAS Version | Type |
|--------|---------------------|-----------------|------|
| USGS ScienceBase | Kalamazoo River, MI | 6.6 | 2D Unsteady |
| Minnesota DNR | Chester Creek | 5.0.3 | 1D Steady |
| Indiana DNR | Fall Creek Trail Bridge | 5.0.7 | 1D Steady |
| Colorado CHAMP | Henson Creek | 4.1.0 | 1D Steady |
| FEMA eBFE/BLE | Spring Creek | 5.0.7 | 2D Unsteady |
| Harris County M3 | Brays Bayou | 3.0.1 | 1D Steady |

**Requirements**: HEC-RAS installed (for RASMapper screenshots), internet connection (for API queries).

In [ ]:
from pathlib import Path
import os
import sys
import time

try:
    from ras_commander import init_ras_project, RasMap
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent))
    from ras_commander import init_ras_project, RasMap

import pandas as pd
from IPython.display import Image, display

from ras_commander.sources.catalog import ModelCatalog, get_catalog
from ras_commander.sources.base import (
    ModelMetadata, ModelType, ModelFilter,
    SourceStatus, DownloadResult,
)
from ras_commander.sources.federal.usgs_sciencebase import UsgsScienceBase
from ras_commander.sources.state.mn_dnr import MinnesotaDnrModels
from ras_commander.sources.state.in_dnr import IndianaDnrModels
from ras_commander.sources.state.co_champ import ColoradoChampModels
from ras_commander.sources.federal.noaa_ras2fim import NoaaRas2fimModels
from ras_commander.sources.federal import RasEbfeModels
from ras_commander.sources.county import M3Model

# Shared workspace
DOWNLOAD_ROOT = Path(os.environ.get(
    "RAS_COMMANDER_MODEL_SOURCES_ROOT",
    r"H:\Testing\Model_Sources_Showcase"
))
SCREENSHOT_DIR = DOWNLOAD_ROOT / "screenshots"
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Download root: {DOWNLOAD_ROOT}")
print(f"Screenshots:   {SCREENSHOT_DIR}")

In [ ]:
def find_ras_prj(folder: Path) -> Path:
    """Find the primary HEC-RAS .prj file in a folder (not CRS projection files)."""
    for prj in sorted(folder.rglob("*.prj")):
        try:
            first_line = prj.read_text(encoding="utf-8", errors="ignore").split("\n")[0]
            if first_line.startswith("Proj Title="):
                return prj
        except Exception:
            continue
    return None


def screenshot_and_display(
    prj_path: Path,
    label: str,
    ras_version: str,
    delay: float = 8.0,
) -> Path:
    """Take a RASMapper screenshot and display it inline."""
    png_path = SCREENSHOT_DIR / f"{label}.png"
    print(f"Capturing RASMapper screenshot for {label}...")
    print(f"  .prj: {prj_path}")
    print(f"  RAS version: {ras_version}")

    try:
        result = RasMap.screenshot_model(
            ras_project_path=prj_path,
            output_path=png_path,
            delay_seconds=delay,
            ras_version=ras_version,
        )
    except FileNotFoundError as exc:
        print(f"  Skipped (no .rasmap file): {exc}")
        return None
    except Exception as exc:
        print(f"  Screenshot failed: {exc}")
        return None

    if result and result.exists() and result.stat().st_size > 0:
        print(f"  Saved: {result} ({result.stat().st_size / 1024:.1f} KB)")
        display(Image(filename=str(result), width=800))
        return result
    else:
        print(f"  Screenshot capture failed.")
        return None


print("Helper functions ready.")

## Unified Catalog

The `ModelCatalog` auto-registers all available sources. Cross-source search with a single call.

In [ ]:
catalog = get_catalog()

print("Registered sources:")
for name in catalog.list_sources(include_unavailable=True):
    status = catalog._source_status.get(name, SourceStatus.UNAVAILABLE)
    print(f"  {name}: {status.value}")

print("\nCross-source search: 'Creek' (limit 3 per source)")
creek_models = catalog.search_models(location="Creek", limit_per_source=3)
print(f"Found {len(creek_models)} models:\n")
for m in creek_models:
    print(f"  [{m.source_name}] {m.name} -- {m.model_type.value}")

---
## Source 1: USGS ScienceBase -- Kalamazoo River, MI

Peer-reviewed 2D unsteady model (HEC-RAS 6.6, ~3 GB). DOI: 10.5066/P13CPA5B.

21 plans with HDF outputs, ADCP velocity and WSE calibration data.

In [ ]:
# List ScienceBase catalog
sb_models = UsgsScienceBase.list_catalog_models()
for m in sb_models:
    print(f"{m.name} -- {m.model_type.value}, v{m.hecras_version}, DOI: {m.doi}")

In [ ]:
# Download Kalamazoo River (USGS ScienceBase) — ~3 GB, may take several minutes
sb_result = UsgsScienceBase.download_model(
    "kalamazoo",
    output_dir=DOWNLOAD_ROOT / "usgs",
)
sb_prj = find_ras_prj(sb_result) if sb_result else None
print(f"USGS prj: {sb_prj}")

In [ ]:
if sb_prj:
    screenshot_and_display(sb_prj, "01_USGS_ScienceBase_Kalamazoo", ras_version="7.0")

---
## Source 2: Minnesota DNR -- Chester Creek

~2,384 FEMA floodplain HEC-RAS models across 67 MN counties, all 1D steady-state.

Chester Creek (St. Louis County) is a multi-geometry model built with HEC-RAS 5.0.3.

> **Note**: MN DNR models are 1D steady-state (HEC-RAS 4.x/5.x), which predate the RASMapper `.rasmap` convention. Screenshots are not available for this source.

In [ ]:
# Query MN DNR catalog
mn_counties = MinnesotaDnrModels.list_counties()
print(f"MN DNR: {len(mn_counties)} counties")

mn_models = MinnesotaDnrModels.list_models(location="St. Louis", limit=5)
print(f"St. Louis County models (up to 5): {len(mn_models)}")
for m in mn_models:
    print(f"  {m.name} -- {m.source_id}")

In [ ]:
# Download Chester Creek (MN DNR)
# Search the full catalog by name — location-based filtering is unreliable
# when the API county name format changes.
mn_all = MinnesotaDnrModels.list_models(limit=100)
chester = [m for m in mn_all if "Chester" in m.name]
mn_prj = None
if chester:
    mn_model = chester[0]
elif mn_models:
    mn_model = mn_models[0]
else:
    mn_model = mn_all[0] if mn_all else None

if mn_model:
    mn_result = MinnesotaDnrModels.download_model(
        mn_model.source_id,
        output_folder=DOWNLOAD_ROOT / "mn_dnr",
    )
    mn_prj = find_ras_prj(mn_result.model_path) if mn_result and mn_result.success else None
    print(f"MN DNR model: {mn_model.name}")
else:
    print("MN DNR: no models available")
print(f"MN DNR prj: {mn_prj}")

In [ ]:
if mn_prj:
    screenshot_and_display(mn_prj, "02_MN_DNR_Chester_Creek", ras_version="7.0")

---
## Source 3: Indiana DNR -- Fall Creek Trail Bridge

~11,400 models across 92 IN counties (largest single state source).

Fall Creek Trail Bridge (Hamilton County) is a 1D steady model with 4 plans, built with HEC-RAS 5.0.7.

> **Note**: IN DNR models are 1D steady-state (HEC-RAS 4.x/5.x), which predate the RASMapper `.rasmap` convention. Screenshots are not available for this source.

In [ ]:
# Query IN DNR catalog
in_counties = IndianaDnrModels.list_counties()
print(f"IN DNR: {len(in_counties)} counties")

in_models = IndianaDnrModels.list_models(location="Hamilton", limit=5)
print(f"Hamilton County models (up to 5): {len(in_models)}")
for m in in_models:
    print(f"  {m.name} -- ID: {m.source_id}, {m.model_type.value}")

In [ ]:
# Download Fall Creek Trail Bridge (IN DNR — Hamilton County)
fall_creek = [m for m in IndianaDnrModels.list_models(location="Hamilton") if "Fall Creek" in m.name]
in_model = fall_creek[0] if fall_creek else in_models[0]
in_result = IndianaDnrModels.download_model(
    in_model.source_id,
    output_folder=DOWNLOAD_ROOT / "in_dnr",
)
in_prj = find_ras_prj(in_result.model_path) if in_result and in_result.success else None
print(f"IN DNR prj: {in_prj}")

In [ ]:
if in_prj:
    screenshot_and_display(in_prj, "03_IN_DNR_Fall_Creek", ras_version="7.0")

---
## Source 4: Colorado CHAMP

Small CWCB CHAMP catalog (~3 downloadable entries). Catalog contents change over time; some entries are non-HEC-RAS studies (HMS hydrology models, shapefiles only). The download cell iterates through available entries until one containing a valid HEC-RAS project is found.

In [ ]:
# Query CHAMP catalog
co_models = ColoradoChampModels.list_models()
print(f"Colorado CHAMP models: {len(co_models)}")
for m in co_models:
    size = f"{m.file_size_mb:.1f} MB" if m.file_size_mb else "unknown"
    print(f"  {m.name} -- {m.model_type.value} ({size})")

In [ ]:
# Download first CO CHAMP entry that contains a HEC-RAS project.
# Some catalog entries are non-HEC-RAS studies (HMS models, shapefiles only);
# iterate until find_ras_prj() succeeds.
co_prj = None
for co_model in co_models:
    co_result = ColoradoChampModels.download_model(
        co_model.source_id,
        output_folder=DOWNLOAD_ROOT / "co_champ",
    )
    if co_result and co_result.success:
        co_prj = find_ras_prj(co_result.model_path)
        if co_prj:
            print(f"CO CHAMP model: {co_model.name}")
            break
        print(f"  {co_model.name}: no HEC-RAS project found, trying next")
if not co_prj:
    print("CO CHAMP: no models with HEC-RAS content found in catalog")
print(f"CO CHAMP prj: {co_prj}")

In [ ]:
if co_prj:
    screenshot_and_display(co_prj, "04_CO_CHAMP_Henson_Creek", ras_version="7.0")

---
## Source 5: FEMA eBFE/BLE -- Spring Creek

Large 2D unsteady models from FEMA's BLE program. Spring Creek (HUC 12040102) is organized via `RasEbfeModels.organize_model()` into the standard delivery format.

See notebooks 950-957 for detailed eBFE workflows.

In [ ]:
# List available eBFE models
seen = set()
for alias, canonical in sorted(RasEbfeModels._MODEL_ALIASES.items()):
    if canonical not in seen:
        seen.add(canonical)
        print(f"  {canonical}")
print(f"\nTotal eBFE models: {len(seen)}")

In [ ]:
# Organize Spring Creek eBFE model (downloads + fixes broken paths automatically)
ebfe_result = RasEbfeModels.download_model(
    "spring-creek",
    output_folder=DOWNLOAD_ROOT / "ebfe",
)
if ebfe_result.success and ebfe_result.model_path:
    ebfe_ras_dir = ebfe_result.model_path / "RAS Model"
    ebfe_prj = find_ras_prj(ebfe_ras_dir) or find_ras_prj(ebfe_result.model_path)
else:
    ebfe_prj = None
print(f"eBFE prj: {ebfe_prj}")

In [ ]:
if ebfe_prj:
    screenshot_and_display(ebfe_prj, "05_eBFE_Spring_Creek", ras_version="7.0")

---
## Source 6: Harris County M3 -- Brays Bayou

20 FEMA-effective H&H models for Houston-area watersheds. All 1D steady (HEC-RAS 3.0.1).

> **Note**: HEC-RAS 3.0.1 predates RASMapper -- no `.rasmap` file is included and screenshots are not available for this source.

In [ ]:
# List M3 models
m3_df = M3Model.list_models(as_dataframe=True)
print(f"Harris County M3 Models: {len(m3_df)}\n")
print(m3_df[['name', 'description', 'effective_date', 'size_gb']].to_string(index=True))

In [ ]:
# Download Brays Bayou (Harris County M3 — model D)
# M3 delivers sub-reach models as nested zips; extract inner archives after outer.
import zipfile as _zipfile

m3_path = M3Model.extract_model("D", output_path=DOWNLOAD_ROOT / "m3")
m3_prj = None
if m3_path:
    for inner_zip in m3_path.rglob("*.zip"):
        dest = inner_zip.parent / inner_zip.stem
        if not dest.exists():
            try:
                with _zipfile.ZipFile(inner_zip, "r") as zf:
                    zf.extractall(dest)
                print(f"  Extracted: {inner_zip.name}")
            except _zipfile.BadZipFile:
                pass
    m3_prj = find_ras_prj(m3_path)
print(f"M3 prj: {m3_prj}")

In [ ]:
if m3_prj:
    screenshot_and_display(m3_prj, "06_M3_Brays_Bayou", ras_version="7.0")

---
## Source 7: NOAA ras2fim (Catalog Only)

NWM-aligned 1D models on S3 (`noaa-nws-owp-fim`). Full access requires `boto3` with AWS credentials; without it, a hardcoded registry is returned.

In [ ]:
noaa_status = NoaaRas2fimModels.get_source_status()
print(f"NOAA ras2fim status: {noaa_status.value}")

try:
    import boto3
    print("boto3: available")
except ImportError:
    print("boto3: not installed (using hardcoded registry)")

noaa_models = NoaaRas2fimModels.list_models(limit=5)
print(f"\nNOAA ras2fim models (up to 5): {len(noaa_models)}")
for m in noaa_models:
    print(f"  {m.name} -- HUC8: {m.location}, {m.model_type.value}")

---
## Screenshot Gallery

Collect all captured screenshots into a summary gallery.

In [ ]:
# Display all captured screenshots
screenshots = sorted(SCREENSHOT_DIR.glob("*.png"))
print(f"Screenshots captured: {len(screenshots)}\n")

for png in screenshots:
    label = png.stem.replace("_", " ")
    size_kb = png.stat().st_size / 1024
    print(f"--- {label} ({size_kb:.0f} KB) ---")
    display(Image(filename=str(png), width=700))
    print()

---
## Summary

| Source | Model | Version | Type | Screenshot |
|--------|-------|---------|------|------------|
| USGS ScienceBase | Kalamazoo River, MI | 6.6 | 2D Unsteady | Yes |
| Minnesota DNR | Chester Creek | 5.0.3 | 1D Steady | No (pre-RASMapper) |
| Indiana DNR | Fall Creek Trail Bridge | 5.0.7 | 1D Steady | No (pre-RASMapper) |
| Colorado CHAMP | First HEC-RAS entry | varies | 1D Steady | Yes (when found) |
| FEMA eBFE/BLE | Spring Creek | 5.0.7 | 2D Unsteady | Yes |
| Harris County M3 | Brays Bayou | 3.0.1 | 1D Steady | No (pre-RASMapper) |
| NOAA ras2fim | (catalog only) | -- | 1D Steady | No (S3 only) |

Sources 1 and 5 (2D models with `.rasmap` files) produce terrain screenshots via `RasMap.screenshot_model()`. Sources 2, 3, and 6 are 1D steady-state models predating RASMapper; source 4 (CO CHAMP) screenshots when a HEC-RAS project is found in the catalog.